# **TamilNadu level Dataset Generation**

In [ ]:
# NumPy is used for random-number generation and numerical calculations.
import numpy as np
# Pandas is used to create and manipulate the applicant dataset.
import pandas as pd
# os is used for creating folders and constructing file paths.
import os

# Fix the random seed so that the same synthetic dataset can be reproduced.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# Create a folder inside the local Google Colab runtime.
# Files stored under /content are available during the current Colab session.
MODEL_DIR = '/content/faid_optimizer'
os.makedirs(MODEL_DIR, exist_ok=True)

# Define the local path where the generated synthetic CSV will be saved.
DATA_PATH = os.path.join(MODEL_DIR, 'tn_synthetic_aid_dataset.csv')
print('Dataset will be saved to:', DATA_PATH)

Dataset will be saved to: /content/faid_optimizer/tn_synthetic_aid_dataset.csv


In [ ]:
# List of Tamil Nadu districts represented in the synthetic applicant pool.
DISTRICTS = ["Chennai","Coimbatore","Madurai","Tiruchirappalli","Salem","Tirunelveli",
             "Erode","Vellore","Thanjavur","Dindigul","Namakkal","Tiruppur","Theni",
             "Cuddalore","Karur","Nagapattinam","Virudhunagar","Kanyakumari","Villupuram","Thoothukudi"]

# Define three synthetic college tiers and five admission categories.
COLLEGE_TIER = ["Tier-1 (Autonomous)", "Tier-2 (Affiliated)", "Tier-3 (Self-financing)"]
CATEGORY = ["OC", "BC", "MBC", "SC", "ST"]

# Approximate category sampling probabilities used only for synthetic-data generation.
CATEGORY_P = [0.12, 0.30, 0.28, 0.20, 0.10]


In [ ]:
# Generate a synthetic applicant dataset with a reproducible random process.
def generate_tn_dataset(n=6000, seed=RANDOM_STATE):
    # Create an independent random-number generator for reproducible results.
    rng = np.random.default_rng(seed)

    # Randomly assign each applicant a district and admission category.
    district = rng.choice(DISTRICTS, n)
    category = rng.choice(CATEGORY, n, p=CATEGORY_P)

    # Generate an urban/rural indicator: 1 = urban, 0 = rural.
    urban = rng.choice([0, 1], n, p=[0.45, 0.55])

    # Set different baseline family-income levels for urban and rural applicants.
    base_income = np.where(urban == 1, 420000, 260000)

    # Apply category-based income multipliers for synthetic calibration.
    cat_adj = pd.Series(category).map({"OC":1.5,"BC":1.15,"MBC":1.0,"SC":0.75,"ST":0.65}).values

    # Generate realistic-looking family income values using a log-normal distribution.
    family_income = np.clip(rng.lognormal(mean=np.log(base_income*cat_adj), sigma=0.55), 60000, 4500000).round(-3)

    # Generate first-generation college status and parent graduation status.
    first_gen = rng.choice([0, 1], n, p=[0.55, 0.45])
    parent_grad = np.where(first_gen == 1, 0, rng.choice([0, 1], n, p=[0.35, 0.65]))

    # Generate academic-strength indicators.
    cutoff_12th = np.clip(rng.normal(78, 10, n), 45, 99.9).round(2)
    entrance_score = np.clip(cutoff_12th*1.6 + rng.normal(0, 15, n), 30, 200).round(1)

    # Assign a college tier and generate tuition according to the tier.
    tier = rng.choice(COLLEGE_TIER, n, p=[0.2, 0.45, 0.35])
    tuition = pd.Series(tier).map({"Tier-1 (Autonomous)":180000, "Tier-2 (Affiliated)":110000, "Tier-3 (Self-financing)":85000}).values
    tuition = (tuition + rng.normal(0, 8000, n)).round(-3).clip(50000)

    # Generate distance from the college and the number of competing offers.
    distance_km = np.clip(rng.exponential(60, n), 2, 600).round(1)
    competing_offers = rng.poisson(1.4, n)

    # Convert income and academic score into normalized need and merit indicators.
    need_index = np.clip(1 - (family_income / family_income.max()), 0.02, 0.98)
    merit_index = np.clip((entrance_score - 30) / 170, 0.02, 0.98)

    # Generate synthetic merit-based and need-based aid percentages.
    merit_aid_pct = np.clip(merit_index*0.6 + rng.normal(0, 0.08, n), 0, 0.7)
    need_aid_pct = np.clip(need_index*0.55 + rng.normal(0, 0.08, n), 0, 0.75)

    # Combine the two aid components into a total aid percentage.
    total_aid_pct = np.clip(merit_aid_pct*0.5 + need_aid_pct*0.5, 0, 0.85)

    # Calculate the monetary aid amount and the resulting net price.
    aid_amount = (tuition * total_aid_pct).round(-2)
    net_price = (tuition - aid_amount).round(-2)

    # Estimate price sensitivity and affordability for the synthetic enrollment model.
    price_sensitivity = np.clip(0.55 + need_index*0.6 - merit_index*0.25, 0.15, 1.4)
    affordability = 1 - np.clip(net_price / (family_income*0.4 + 1), 0, 1.5)

    # Build a synthetic enrollment logit from affordability, aid burden, merit,
    # competing offers, distance, parent education, and random variation.
    logit = (
        -0.6
        + 2.6*affordability
        - 1.1*price_sensitivity*(net_price/tuition)
        + 0.9*merit_index
        - 0.18*competing_offers
        - 0.15*(distance_km/600)
        + 0.35*parent_grad
        + rng.normal(0, 0.5, n)
    )

    # Convert the logit score to an enrollment probability using the sigmoid function.
    prob_enroll = 1 / (1 + np.exp(-logit))

    # Sample the final binary enrollment label from the calculated probability.
    enrolled = rng.binomial(1, prob_enroll)

    # Assemble all generated variables into a Pandas DataFrame.
    df = pd.DataFrame({
        "district": district, "category": category, "urban": urban,
        "family_income": family_income, "first_gen": first_gen, "parent_grad": parent_grad,
        "cutoff_12th": cutoff_12th, "entrance_score": entrance_score,
        "college_tier": tier, "tuition": tuition, "distance_km": distance_km,
        "competing_offers": competing_offers, "merit_aid_pct": merit_aid_pct.round(3),
        "need_aid_pct": need_aid_pct.round(3), "total_aid_pct": total_aid_pct.round(3),
        "aid_amount": aid_amount, "net_price": net_price, "enrolled": enrolled
    })

    # Return the completed synthetic applicant dataset.
    return df


In [ ]:
# Generate 6,000 synthetic applicant records using the fixed random seed.
# Using the same seed makes the generated dataset reproducible.
df = generate_tn_dataset(n=6000, seed=RANDOM_STATE)

# Save the generated dataset as a CSV file in the local Colab runtime.
df.to_csv(DATA_PATH, index=False)

# Display basic information so we can verify that generation and saving worked.
print('Saved ->', DATA_PATH)
print('Shape:', df.shape)
print('Overall yield rate:', round(df['enrolled'].mean(), 3))
df.head()


Saved -> /content/faid_optimizer/tn_synthetic_aid_dataset.csv
Shape: (6000, 18)
Overall yield rate: 0.563


,district,category,urban,family_income,first_gen,parent_grad,cutoff_12th,entrance_score,college_tier,tuition,distance_km,competing_offers,merit_aid_pct,need_aid_pct,total_aid_pct,aid_amount,net_price,enrolled
0,Coimbatore,SC,1,691000.0,0,1,92.35,145.6,Tier-2 (Affiliated),110000.0,29.8,3,0.443,0.430,0.437,48000.0,62000.0,1
1,Nagapattinam,MBC,0,686000.0,1,0,77.63,130.7,Tier-3 (Self-financing),91000.0,15.8,0,0.389,0.324,0.357,32500.0,58500.0,1
2,Cuddalore,BC,1,579000.0,0,1,68.22,104.2,Tier-2 (Affiliated),108000.0,66.9,0,0.292,0.422,0.357,38500.0,69500.0,1
3,Thanjavur,SC,0,99000.0,0,0,80.98,129.6,Tier-3 (Self-financing),91000.0,31.2,1,0.504,0.457,0.481,43800.0,47200.0,0
4,Thanjavur,OC,0,1034000.0,1,0,77.28,126.9,Tier-2 (Affiliated),119000.0,19.4,2,0.395,0.389,0.392,46600.0,72400.0,0


In [ ]:
# Display the null or missing values in the dataset
print(df.isnull().sum().sum(), 'missing values')
print()

# Display the datatype of each columns in the dataset
print(df.dtypes)
print()

# Dispaly the distribution of each category
print('Category distribution:')
print(df['category'].value_counts(normalize=True).round(3))
print()

# Display the yield rate by college tier wise
print('Yield rate by college tier:')
print(df.groupby('college_tier')['enrolled'].mean().round(3))

0 missing values

district             object
category             object
urban                 int64
family_income       float64
first_gen             int64
parent_grad           int64
cutoff_12th         float64
entrance_score      float64
college_tier         object
tuition             float64
distance_km         float64
competing_offers      int64
merit_aid_pct       float64
need_aid_pct        float64
total_aid_pct       float64
aid_amount          float64
net_price           float64
enrolled              int64
dtype: object

Category distribution:
category
BC     0.308
MBC    0.283
SC     0.195
OC     0.117
ST     0.097
Name: proportion, dtype: float64

Yield rate by college tier:
college_tier
Tier-1 (Autonomous)        0.437
Tier-2 (Affiliated)        0.566
Tier-3 (Self-financing)    0.629
Name: enrolled, dtype: float64


# **US DataSet Generation**

In [ ]:
# NumPy is used for random-number generation and numerical calculations.
import numpy as np

# Pandas is used to create, organize, and manipulate the applicant dataset.
import pandas as pd

# Google Colab's files module is used to download the generated dataset.
from google.colab import files

In [ ]:
# Approximate relative population weights for U.S. states.
# These weights are used to create a non-uniform distribution of applicants
# across the 50 states.
STATE_WEIGHTS = {
    "CA": 0.118, "TX": 0.089, "FL": 0.065, "NY": 0.058, "PA": 0.038,
    "IL": 0.037, "OH": 0.035, "GA": 0.032, "NC": 0.031, "MI": 0.030,
    "NJ": 0.027, "VA": 0.026, "WA": 0.023, "AZ": 0.022, "MA": 0.021,
    "TN": 0.021, "IN": 0.020, "MO": 0.018, "MD": 0.018, "WI": 0.018,
    "CO": 0.017, "MN": 0.017, "SC": 0.016, "AL": 0.015, "LA": 0.014,
    "KY": 0.013, "OR": 0.013, "OK": 0.012, "CT": 0.011, "UT": 0.010,
    "IA": 0.010, "NV": 0.009, "AR": 0.009, "MS": 0.009, "KS": 0.009,
    "NM": 0.006, "NE": 0.006, "WV": 0.005, "ID": 0.005, "HI": 0.004,
    "NH": 0.004, "ME": 0.004, "MT": 0.003, "RI": 0.003, "DE": 0.003,
    "SD": 0.003, "ND": 0.002, "AK": 0.002, "VT": 0.002, "WY": 0.002,
}

# Create a list containing all U.S. states in the synthetic dataset.
STATES = list(STATE_WEIGHTS.keys())

# Convert the state weights into a NumPy array for random sampling.
STATE_P = np.array(list(STATE_WEIGHTS.values()))

# Normalize the state probabilities so that their total is exactly 1.
STATE_P = STATE_P / STATE_P.sum()


# Define the institution's home state.
# This is used to distinguish between in-state and out-of-state applicants.
HOME_STATE = "OH"


# Define geographic location categories for applicants.
LOCALE_CATEGORIES = [
    "City",
    "Suburb",
    "Town",
    "Rural"
]

# Approximate probability of each geographic location category.
LOCALE_P = [0.30, 0.40, 0.15, 0.15]


# Define the institutional categories used in the synthetic dataset.
TIER_CATEGORIES = [
    "R1 Research",
    "R2 Doctoral",
    "Regional Comprehensive",
    "Liberal Arts College",
    "Community College",
]

# Approximate probability of each institutional category.
TIER_P = [0.18, 0.14, 0.34, 0.14, 0.20]


# Define the approximate base cost of attendance for each
# institutional category.
TIER_BASE_COA = {
    "R1 Research": 58000,
    "R2 Doctoral": 42000,
    "Regional Comprehensive": 31000,
    "Liberal Arts College": 63000,
    "Community College": 19000,
}


# Define the standard deviation used to add realistic variation
# to the cost of attendance for each institutional category.
TIER_COA_SD = {
    "R1 Research": 9000,
    "R2 Doctoral": 6000,
    "Regional Comprehensive": 4500,
    "Liberal Arts College": 10000,
    "Community College": 2500,
}


# Additional cost applied to eligible out-of-state applicants.
OUT_OF_STATE_PREMIUM = 14000

# Minimum possible cost of attendance allowed in the dataset.
RNG_MIN_COA = 8000

# cap the maximum share of COA that aid can cover. Real institutions
# essentially never package a student to exactly 100% of COA outside a
# deliberate free-tuition program, so an unconstrained sum could otherwise
# produce implausible FADR values right at 1.0. 0.95 leaves room for the
# rare near-full-ride case without letting it become a systematic artifact.
MAX_AID_SHARE_OF_COA = 0.95

# tighten the NPIR cap. A ratio above ~3.0 (net price triple a
# family's entire annual income) is already an extreme-outlier regime;
# capping here keeps the raw column usable without needing a hard
# downstream winsorization step just to make the column plottable.
NPIR_CAP = 3.0

# instead of hard-clipping implausible low test scores to a
# placeholder "1st percentile" (which piles many dissimilar students up on
# an identical value and reads like missing-data-as-zero), any score that
# would fall below this plausible floor is treated as genuinely missing
# (NaN) — mirroring real test-optional non-submission.
SAT_PLAUSIBLE_FLOOR = 1.0

In [ ]:
# The sigmoid function converts a numerical score into a probability
# between 0 and 1. It is used to calculate enrollment probability.
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

In [ ]:
# This function generates the complete synthetic financial-aid
# and enrollment dataset.
def generate_dataset(n=10000, seed=42):

    # Create a random-number generator using the specified seed.
    # Using the same seed makes the generated dataset reproducible.
    rng = np.random.default_rng(seed)

    # Geography

    # Randomly assign a U.S. state of residence to each applicant
    # according to the predefined state probabilities.
    state_residency = rng.choice(
        STATES,
        size=n,
        p=STATE_P
    )

    # Randomly assign a geographic location category to each applicant.
    urban_centric_locale = rng.choice(
        LOCALE_CATEGORIES,
        size=n,
        p=LOCALE_P
    )

    # Identify whether each applicant belongs to the institution's
    # home state.
    is_in_state = (state_residency == HOME_STATE)

    # Generate the basic distance from the applicant's home to campus.
    # A log-normal distribution creates a realistic right-skewed distance.
    base_miles = rng.lognormal(
        mean=4.2,
        sigma=1.05,
        size=n
    )

    # Calculate final distance from campus.
    # In-state students are generally closer, while out-of-state
    # students are allowed to have substantially greater distances.
    miles_from_campus = np.where(
        is_in_state,
        np.clip(base_miles * 0.55, 2, 600),
        np.clip(base_miles * 1.9, 15, 3000),
    ).round(1)

    # Household financial profile

    # Generate adjusted gross household income using a log-normal
    # distribution to represent income variation among households.
    adjusted_gross_income = rng.lognormal(
        mean=11.05,
        sigma=0.72,
        size=n
    )

    # Restrict household income to a reasonable maximum and round
    # the values to whole dollars.
    adjusted_gross_income = np.clip(
        adjusted_gross_income,
        0,
        650_000
    ).round(0)

    # Generate a family-size factor.
    # Family size affects the estimated Student Aid Index (SAI).
    family_size_factor = rng.normal(
        loc=1.0,
        scale=0.18,
        size=n
    )

    # Restrict the family-size factor to a realistic range.
    family_size_factor = np.clip(
        family_size_factor,
        0.55,
        1.6
    )

    # Calculate the base Student Aid Index using household income
    # and the family-size factor.
    sai_raw = (
        adjusted_gross_income * 0.22
    ) / family_size_factor

    # Add random variation to the Student Aid Index.
    sai_noise = rng.normal(
        loc=0,
        scale=3500,
        size=n
    )

    # Calculate the final Student Aid Index and restrict it
    # to the defined range.
    # NOTE (Issue not a bug): negative SAI values as low as -1500 are
    # kept deliberately. Under the FAFSA Simplification Act, -1500 is the
    # real federal floor and signals extreme financial need — it should
    # stay in the data. Just don't run a [0,1]-bounded scaler or a log
    # transform directly on this column in your preprocessing pipeline.
    student_aid_index = np.clip(
        sai_raw + sai_noise - 1800,
        -1500,
        220_000
    ).round(0)


    # Financial-need indicators

    # Calculate an SAI threshold used to generate Pell Grant eligibility.
    pell_threshold = np.quantile(
        student_aid_index,
        0.34
    )

    # Mark applicants as Pell eligible when their SAI is at or below
    # the calculated threshold.
    pell_eligible = (
        student_aid_index <= pell_threshold
    ).astype(int)

    # Calculate the probability of being a first-generation
    # college student based partly on household income.
    first_gen_logit = (
        -0.55
        + (-1.15e-5) * (adjusted_gross_income - 60000)
    )

    # Convert the first-generation score into a probability.
    first_gen_p = np.clip(
        sigmoid(first_gen_logit),
        0.05,
        0.85
    )

    # Generate the first-generation indicator.
    # 1 represents a first-generation student and 0 represents otherwise.
    first_gen = rng.binomial(
        1,
        first_gen_p
    )

    # Academic profile

    # Generate high-school GPA using a beta distribution.
    # This produces values concentrated toward the higher end of the GPA range.
    gpa_beta = rng.beta(
        a=6.2,
        b=2.3,
        size=n
    )

    # Add small adjustments based on first-generation status
    # and household income.
    gpa_adj = (
        0.06 * (1 - first_gen)
        + 0.00000025 * (adjusted_gross_income - 60000)
    )

    # Convert the generated value to a 4.0 GPA scale and
    # restrict it to the range 1.5-4.0.
    hs_gpa = np.clip(
        gpa_beta * 4.0 + gpa_adj,
        1.5,
        4.0
    ).round(2)


    # Continuous "true ability" score used internally for every downstream
    # calculation (merit aid, overqualification, enrollment logit) — this
    # never contains NaN, so the rest of the generator behaves exactly as
    # before regardless of what gets reported in the public column.
    # This creates a moderate correlation between GPA and test performance.
    sat_core = (
        55
        + (hs_gpa - 3.0) * 38
        + rng.normal(0, 13, size=n)
    )
    sat_core_internal = np.clip(sat_core, 1, 99)

    # Restrict standardized-test percentile to the range 1-99.
    #the *reported* column is what changes. Instead of clipping
    # every low value to a placeholder "1", any score that would fall below
    # the plausible floor is reported as missing (NaN) — representing a
    # real test-optional non-submission rather than a fabricated data point.
    sat_act_percentile = np.where(sat_core < SAT_PLAUSIBLE_FLOOR, np.nan, sat_core)
    sat_act_percentile = np.clip(sat_act_percentile, 1, 99)
    sat_act_percentile = np.round(sat_act_percentile, 0)

    # Institutional assignment and cost

    # Randomly assign each applicant to one of the institutional tiers.
    institutional_tier = rng.choice(
        TIER_CATEGORIES,
        size=n,
        p=TIER_P
    )

    # Retrieve the base cost of attendance for each applicant's
    # assigned institutional tier.
    base_coa = np.array([
        TIER_BASE_COA[t]
        for t in institutional_tier
    ])


    # Retrieve the cost variation associated with each tier.
    coa_sd = np.array([
        TIER_COA_SD[t]
        for t in institutional_tier
    ])


    # Generate random variation in cost of attendance.
    coa_noise = (
        rng.normal(0, 1, size=n)
        * coa_sd
    )


    # Add the out-of-state premium when applicable.
    oos_premium = np.where(
        (~is_in_state)
        & (institutional_tier != "Community College"),
        OUT_OF_STATE_PREMIUM,
        0
    )


    # Calculate the final cost of attendance.
    cost_of_attendance = np.clip(
        base_coa + coa_noise + oos_premium,
        RNG_MIN_COA,
        None
    ).round(0)

    # Behavioral and engagement profile

    # Define the probability of FAFSA submission for each month.
    # The distribution is intentionally non-uniform to represent
    # different levels of application engagement.
    month_weights = np.array([
        0.20, 0.16, 0.12, 0.09, 0.07, 0.05,
        0.04, 0.04, 0.05, 0.07, 0.05, 0.06
    ])

    # Normalize the monthly probabilities so that their total is 1.
    month_weights = (
        month_weights /
        month_weights.sum()
    )

    # Generate FAFSA submission month from 1 to 12.
    fafsa_submitted_month = rng.choice(
        np.arange(1, 13),
        size=n,
        p=month_weights
    )

    # cyclical sine/cosine encoding, computed right alongside the
    # raw month so both are available in the exported dataset. December
    # (12) and January (1) now sit close together in sin/cos space instead
    # of being treated as numerically far apart.
    fafsa_month_sin = np.sin(2 * np.pi * fafsa_submitted_month / 12).round(4)
    fafsa_month_cos = np.cos(2 * np.pi * fafsa_submitted_month / 12).round(4)

    # Generate the base demonstrated-interest score.
    engagement_base = rng.normal(
        loc=52,
        scale=18,
        size=n
    )

    # Earlier FAFSA submission increases the engagement score slightly.
    engagement_base += (
        13 - fafsa_submitted_month
    ).clip(0, None) * 0.9


    # Add a small engagement adjustment for first-generation students.
    engagement_base += first_gen * 4

    # Restrict demonstrated interest to a 0-100 scale.
    demonstrated_interest = np.clip(
        engagement_base,
        0,
        100
    ).round(1)


    # Financial-aid awarding logic
    # Calculate academic strength from GPA and SAT/ACT percentile.
    # Academic strength uses the internal (never-missing) test score so
    # aid/enrollment logic is unaffected by whether a score was "reported".
    # This mirrors how a real institution still evaluates a test-optional
    # applicant on GPA rather than treating a missing score as a zero.
    sat_component = sat_core_internal / 100.0
    academic_strength = (
        (hs_gpa / 4.0) * 0.5
        + sat_component * 0.5
    )

    # Define how generous each institutional tier is with merit aid.
    tier_merit_generosity = np.array([
        {
            "R1 Research": 0.9,
            "R2 Doctoral": 1.15,
            "Regional Comprehensive": 1.35,
            "Liberal Arts College": 1.25,
            "Community College": 0.4
        }[t]
        for t in institutional_tier
    ])

    # Calculate merit scholarship amounts based on academic strength,
    # cost of attendance, and institutional tier.
    merit_scholarship_amt = np.clip(
        academic_strength
        * 0.42
        * cost_of_attendance
        * tier_merit_generosity
        + rng.normal(0, 1500, size=n),
        0,
        cost_of_attendance * 0.9
    ).round(0)

    # Calculate the estimated financial need gap using
    # cost of attendance and the Student Aid Index.
    need_gap_proxy = np.clip(
        cost_of_attendance - student_aid_index,
        0,
        None
    )

    # Set different need-based grant generosity levels
    # depending on Pell eligibility.
    need_generosity = np.where(
        pell_eligible == 1,
        0.55,
        0.28
    )

    # Calculate the need-based grant amount.
    need_grant_amt = np.clip(
        need_gap_proxy
        * need_generosity
        * rng.uniform(0.5, 1.0, size=n),
        0,
        cost_of_attendance * 0.85
    ).round(0)


    # cap the combined package below full COA (see
    # MAX_AID_SHARE_OF_COA) instead of allowing it to reach exactly 100%.
    # Calculate the total financial-aid package by adding
    # merit scholarship and need-based grant.
    total_aid_package = np.clip(
        merit_scholarship_amt + need_grant_amt,
        0, cost_of_attendance * MAX_AID_SHARE_OF_COA,
    ).round(0)

    # Calculate the net price that the student is expected to pay
    # after receiving financial aid.
    net_price = np.clip(
        cost_of_attendance - total_aid_package,
        0,
        None
    ).round(0)

    # Derived financial-aid features
    # Calculate the Net Price to Income Ratio (NPIR).
    # This represents the remaining education cost relative to household income.
    npir = (
        cost_of_attendance - total_aid_package
    ) / np.clip(
        adjusted_gross_income,
        1000,
        None
    )

    # Restrict NPIR to a reasonable range and round the result.
    npir = np.clip(
        npir,
        0,
        5
    ).round(4)

    # Calculate the Financial Aid Discount Rate (FADR).
    # It represents the proportion of the cost of attendance
    # covered by financial aid.
    fadr = (
        total_aid_package /
        cost_of_attendance
    ).round(4)

    # Calculate the Unmet Financial Need Gap (UFNG).
    # It represents the remaining need after considering the
    # Student Aid Index and financial-aid package.
    # floor the unmet-need gap at 0. A university legally cannot
    # over-package a student past the cost of attendance, so a negative
    # "gap" is a contradiction in terms — it should read as "fully met", 0.
    ufng = (
        cost_of_attendance
        - (student_aid_index + total_aid_package)
    )
    ufng = np.clip(ufng, 0, None).round(0)

    # Generate the number of days available for the engagement period.
    days_to_decision = rng.integers(
        30,
        150,
        size=n
    )

    # Generate the number of applicant touchpoints based on
    # demonstrated-interest level.
    touchpoints = np.clip(
        rng.poisson(
            lam=demonstrated_interest / 8.0,
            size=n
        ),
        0,
        None
    )

    # Calculate Engagement Velocity as the number of touchpoints
    # relative to the available decision period.
    engagement_velocity = (
        touchpoints /
        days_to_decision
    ).round(4)

    # Enrollment probability
    # Assign a synthetic prestige score to each institutional tier.
    tier_prestige = np.array([
        {
            "R1 Research": 0.85,
            "R2 Doctoral": 0.55,
            "Regional Comprehensive": 0.25,
            "Liberal Arts College": 0.65,
            "Community College": 0.05
        }[t]
        for t in institutional_tier
    ])

    # Calculate overqualification by comparing academic strength
    # with the institution's prestige level.
    overqualification = np.clip(
        academic_strength - tier_prestige,
        -1,
        1
    )

    # Calculate the enrollment logit using financial, academic,
    # geographic, and engagement-related factors.
    # NOTE: the logit still uses a *floored* copy of UFNG internally, same
    # as the stored column now — kept explicit here so the enrollment math
    # and the exported UFNG column always agree with each other.
    logit = (
        1.55
        - 3.1 * npir
        - 0.0000045 * ufng
        + 0.021 * (demonstrated_interest - 50)
        - 0.0016 * miles_from_campus
        - 1.35 * overqualification
        + 0.55 * fadr
        + 0.15 * pell_eligible * (need_grant_amt > 0)
        + rng.normal(0, 0.55, size=n)
    )

    # Convert the enrollment score into a probability between 0 and 1.
    enroll_prob = sigmoid(logit)

    # Generate the final enrollment outcome.
    # 1 means the applicant enrolled and 0 means the applicant did not enroll.
    enrolled = rng.binomial(
        1,
        enroll_prob
    )

    # Assemble the final dataset

    # Create a Pandas DataFrame containing the base applicant features,
    # financial-aid variables, derived features, and enrollment target.
    df = pd.DataFrame({
        "state_residency": state_residency,
        "is_in_state": is_in_state.astype(int),
        "urban_centric_locale": urban_centric_locale,
        "student_aid_index": student_aid_index,
        "adjusted_gross_income": adjusted_gross_income,
        "first_gen": first_gen,
        "pell_eligible": pell_eligible,
        "hs_gpa": hs_gpa,
        "sat_act_percentile": sat_act_percentile,
        "institutional_tier": institutional_tier,
        "cost_of_attendance": cost_of_attendance,
        "miles_from_campus": miles_from_campus,
        "fafsa_submitted_month": fafsa_submitted_month,
        "fafsa_month_sin": fafsa_month_sin,
        "fafsa_month_cos": fafsa_month_cos,
        "demonstrated_interest": demonstrated_interest,
        "merit_scholarship_amt": merit_scholarship_amt,
        "need_grant_amt": need_grant_amt,
        "total_aid_package": total_aid_package,
        "net_price": net_price,
        "enrolled": enrolled,

        # Derived features calculated from the generated applicant data.
        "net_price_to_income_ratio": npir,
        "financial_aid_discount_rate": fadr,
        "unmet_financial_need_gap": ufng,
        "engagement_velocity": engagement_velocity,
    })

    # Return the completed synthetic dataset.
    return df

In [ ]:
def build_model_ready_dataset(df):

    model_df = df.copy()

    # 1. Drop absolute direct leakage columns and raw linear date tags
    cols_to_drop = ["total_aid_package", "state_residency", "fafsa_submitted_month"]
    model_df = model_df.drop(columns=[col for col in cols_to_drop if col in model_df.columns])

    # 2. Convert high-cardinality text blocks into explicit numeric representations
    tier_mapping = {
        "Community College": 0,
        "Regional Comprehensive": 1,
        "R2 Doctoral": 2,
        "R1 Research": 3,
        "Liberal Arts College": 4
    }
    if "institutional_tier" in model_df.columns:
        model_df["institutional_tier"] = model_df["institutional_tier"].map(tier_mapping)

    # One-Hot Encode low-cardinality geographic segments safely
    if "urban_centric_locale" in model_df.columns:
        model_df = pd.get_dummies(model_df, columns=["urban_centric_locale"], drop_first=True, dtype=int)

    # 3. Stabilize extreme financial ratios to protect linear and distance-based vectors
    if "net_price_to_income_ratio" in model_df.columns:
        model_df["net_price_to_income_ratio"] = np.log1p(model_df["net_price_to_income_ratio"])

    # sat_act_percentile NaNs are left as-is deliberately; if your
    # model can't handle NaN natively, impute per institutional tier
    # instead of with a single global constant:
    # model_df["sat_act_percentile"] = model_df.groupby("institutional_tier")[
    # "sat_act_percentile"].transform(lambda s: s.fillna(s.median()))

    return model_df


In [ ]:
# Define the number of synthetic student records to generate.
N = 6000

# Define the random seed so that the generated dataset can be reproduced.
SEED = 42

# Generate the synthetic financial-aid yield dataset.
df = generate_dataset(
    n=N,
    seed=SEED
)
model_df = build_model_ready_dataset(df)

In [ ]:
RAW_OUTPUT_PATH = "/content/financial_aid_yield_dataset.csv"
MODEL_OUTPUT_PATH = "/content/financial_aid_yield_dataset_model_ready.csv"

df.to_csv(RAW_OUTPUT_PATH, index=False)
model_df.to_csv(MODEL_OUTPUT_PATH, index=False)

# Display a confirmation message after successfully generating the dataset.
print("Dataset generated successfully!")

# Display the exact location where the CSV file was saved.
print("Raw dataset saved at:", RAW_OUTPUT_PATH)
print("Model-ready dataset saved at:", MODEL_OUTPUT_PATH)

Dataset generated successfully!
Raw dataset saved at: /content/financial_aid_yield_dataset.csv
Model-ready dataset saved at: /content/financial_aid_yield_dataset_model_ready.csv


In [ ]:
# Display the first five records of the dataset.
print("\nFirst five raw records:")
display(df.head())


First five raw records:


,state_residency,is_in_state,urban_centric_locale,student_aid_index,adjusted_gross_income,first_gen,pell_eligible,hs_gpa,sat_act_percentile,institutional_tier,...,demonstrated_interest,merit_scholarship_amt,need_grant_amt,total_aid_package,net_price,enrolled,net_price_to_income_ratio,financial_aid_discount_rate,unmet_financial_need_gap,engagement_velocity
0,CO,0,Suburb,47307.0,206264.0,0,0,3.61,70.0,Regional Comprehensive,...,40.0,22292.0,820.0,23112.0,27342.0,1,0.1326,0.4581,0.0,0.2286
1,OH,1,City,3408.0,28634.0,0,1,3.82,93.0,Community College,...,48.4,1404.0,8103.0,9507.0,9287.0,1,0.3243,0.5059,5879.0,0.0470
2,OR,0,Suburb,4787.0,35389.0,0,1,3.14,65.0,R1 Research,...,20.8,23787.0,33635.0,57422.0,27407.0,1,0.7744,0.6769,22620.0,0.0079
3,IN,0,City,10077.0,40054.0,1,0,2.37,41.0,Regional Comprehensive,...,70.1,15334.0,6499.0,21833.0,31364.0,0,0.7830,0.4104,21287.0,0.0759
4,CA,0,City,16174.0,46827.0,0,0,2.12,32.0,Community College,...,57.7,2764.0,803.0,3567.0,16901.0,0,0.3609,0.1743,727.0,0.0775


In [ ]:
# Display the first five records of the model ready dataset.
print("\nFirst five model ready records:")
display(model_df.head())


First five model ready records:


,is_in_state,student_aid_index,adjusted_gross_income,first_gen,pell_eligible,hs_gpa,sat_act_percentile,institutional_tier,cost_of_attendance,miles_from_campus,...,need_grant_amt,net_price,enrolled,net_price_to_income_ratio,financial_aid_discount_rate,unmet_financial_need_gap,engagement_velocity,urban_centric_locale_Rural,urban_centric_locale_Suburb,urban_centric_locale_Town
0,0,47307.0,206264.0,0,0,3.61,70.0,1,50454.0,169.8,...,820.0,27342.0,1,0.124516,0.4581,0.0,0.2286,0,1,0
1,1,3408.0,28634.0,0,1,3.82,93.0,0,18794.0,10.3,...,8103.0,9287.0,1,0.280884,0.5059,5879.0,0.0470,0,0,0
2,0,4787.0,35389.0,0,1,3.14,65.0,3,84829.0,82.8,...,33635.0,27407.0,1,0.573462,0.6769,22620.0,0.0079,0,1,0
3,0,10077.0,40054.0,1,0,2.37,41.0,1,53197.0,64.8,...,6499.0,31364.0,0,0.578297,0.4104,21287.0,0.0759,0,0,0
4,0,16174.0,46827.0,0,0,2.12,32.0,0,20468.0,45.0,...,803.0,16901.0,0,0.308146,0.1743,727.0,0.0775,0,0,0


In [ ]:
# Display the total number of rows and columns.
print("Raw dataset shape:", df.shape)
print("Model-ready dataset shape:", model_df.shape)

Raw dataset shape: (6000, 25)
Model-ready dataset shape: (6000, 24)


In [ ]:
# Display the names of all dataset columns.
print("\nDataset columns:")
print(df.columns.tolist())

print("\nDataset columns (model-ready):")
print(model_df.columns.tolist())


Dataset columns:
['state_residency', 'is_in_state', 'urban_centric_locale', 'student_aid_index', 'adjusted_gross_income', 'first_gen', 'pell_eligible', 'hs_gpa', 'sat_act_percentile', 'institutional_tier', 'cost_of_attendance', 'miles_from_campus', 'fafsa_submitted_month', 'fafsa_month_sin', 'fafsa_month_cos', 'demonstrated_interest', 'merit_scholarship_amt', 'need_grant_amt', 'total_aid_package', 'net_price', 'enrolled', 'net_price_to_income_ratio', 'financial_aid_discount_rate', 'unmet_financial_need_gap', 'engagement_velocity']

Dataset columns (model-ready):
['is_in_state', 'student_aid_index', 'adjusted_gross_income', 'first_gen', 'pell_eligible', 'hs_gpa', 'sat_act_percentile', 'institutional_tier', 'cost_of_attendance', 'miles_from_campus', 'fafsa_month_sin', 'fafsa_month_cos', 'demonstrated_interest', 'merit_scholarship_amt', 'need_grant_amt', 'net_price', 'enrolled', 'net_price_to_income_ratio', 'financial_aid_discount_rate', 'unmet_financial_need_gap', 'engagement_velocity',

In [ ]:
# Display basic statistics for numerical features.
print("\nDataset summary:")
display(df.describe())

print("\nDataset summary (model-ready):")
display(model_df.describe())


Dataset summary:


,is_in_state,student_aid_index,adjusted_gross_income,first_gen,pell_eligible,hs_gpa,sat_act_percentile,cost_of_attendance,miles_from_campus,fafsa_submitted_month,...,demonstrated_interest,merit_scholarship_amt,need_grant_amt,total_aid_package,net_price,enrolled,net_price_to_income_ratio,financial_aid_discount_rate,unmet_financial_need_gap,engagement_velocity
count,6000.000000,6000.000000,6000.000000,6000.000000,6000.000000,6000.000000,5850.000000,6000.000000,6000.000000,6000.000000,...,6000.000000,6000.000000,6000.000000,6000.000000,6000.000000,6000.000000,6000.000000,6000.000000,6000.00000,6000.000000
mean,0.039000,17080.788667,82976.362167,0.339333,0.340000,2.970050,55.334017,49730.427500,216.304017,4.881167,...,60.203900,14920.278500,10277.548167,25178.404667,24552.022833,0.515833,0.421235,0.471537,10385.52900,0.102219
std,0.193611,16043.429559,65640.372919,0.473523,0.473748,0.565874,22.904805,20919.323773,283.975482,3.625452,...,18.130103,9132.079445,9002.811699,15359.518487,12546.341677,0.499791,0.351679,0.207425,10335.13877,0.073422
min,0.000000,-1500.000000,4385.000000,0.000000,0.000000,1.500000,1.000000,10733.000000,3.100000,1.000000,...,0.000000,0.000000,0.000000,0.000000,1285.000000,0.000000,0.016900,0.000000,0.00000,0.000000
25%,0.000000,6704.500000,39297.750000,0.000000,0.000000,2.590000,39.000000,38617.250000,60.400000,2.000000,...,47.600000,7893.500000,3832.750000,14241.000000,15138.500000,0.000000,0.207800,0.338625,1048.50000,0.052600
50%,0.000000,12897.500000,64302.500000,0.000000,0.000000,3.050000,56.000000,48581.500000,122.200000,4.000000,...,60.200000,15508.500000,7891.500000,25076.000000,22090.500000,1.000000,0.327200,0.475400,7644.50000,0.082400
75%,0.000000,22413.750000,104686.750000,1.000000,1.000000,3.400000,72.000000,65720.500000,252.900000,8.000000,...,72.700000,20902.500000,14922.250000,34714.250000,32221.000000,1.000000,0.518900,0.607800,16419.75000,0.129675
max,1.000000,150773.000000,650000.000000,1.000000,1.000000,4.000000,99.000000,107801.000000,3000.000000,12.000000,...,100.000000,51882.000000,52366.000000,98036.000000,75564.000000,1.000000,5.000000,0.950000,63589.00000,0.645200



Dataset summary (model-ready):


,is_in_state,student_aid_index,adjusted_gross_income,first_gen,pell_eligible,hs_gpa,sat_act_percentile,institutional_tier,cost_of_attendance,miles_from_campus,...,need_grant_amt,net_price,enrolled,net_price_to_income_ratio,financial_aid_discount_rate,unmet_financial_need_gap,engagement_velocity,urban_centric_locale_Rural,urban_centric_locale_Suburb,urban_centric_locale_Town
count,6000.000000,6000.000000,6000.000000,6000.000000,6000.000000,6000.000000,5850.000000,6000.000000,6000.000000,6000.000000,...,6000.000000,6000.000000,6000.000000,6000.000000,6000.000000,6000.00000,6000.000000,6000.000000,6000.000000,6000.000000
mean,0.039000,17080.788667,82976.362167,0.339333,0.340000,2.970050,55.334017,1.688333,49730.427500,216.304017,...,10277.548167,24552.022833,0.515833,0.328999,0.471537,10385.52900,0.102219,0.145833,0.402833,0.153000
std,0.193611,16043.429559,65640.372919,0.473523,0.473748,0.565874,22.904805,1.335226,20919.323773,283.975482,...,9002.811699,12546.341677,0.499791,0.199539,0.207425,10335.13877,0.073422,0.352968,0.490509,0.360018
min,0.000000,-1500.000000,4385.000000,0.000000,0.000000,1.500000,1.000000,0.000000,10733.000000,3.100000,...,0.000000,1285.000000,0.000000,0.016759,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,6704.500000,39297.750000,0.000000,0.000000,2.590000,39.000000,1.000000,38617.250000,60.400000,...,3832.750000,15138.500000,0.000000,0.188801,0.338625,1048.50000,0.052600,0.000000,0.000000,0.000000
50%,0.000000,12897.500000,64302.500000,0.000000,0.000000,3.050000,56.000000,1.000000,48581.500000,122.200000,...,7891.500000,22090.500000,1.000000,0.283071,0.475400,7644.50000,0.082400,0.000000,0.000000,0.000000
75%,0.000000,22413.750000,104686.750000,1.000000,1.000000,3.400000,72.000000,3.000000,65720.500000,252.900000,...,14922.250000,32221.000000,1.000000,0.417986,0.607800,16419.75000,0.129675,0.000000,1.000000,0.000000
max,1.000000,150773.000000,650000.000000,1.000000,1.000000,4.000000,99.000000,4.000000,107801.000000,3000.000000,...,52366.000000,75564.000000,1.000000,1.791759,0.950000,63589.00000,0.645200,1.000000,1.000000,1.000000


In [ ]:
# Display important dataset statistics.
print(f"\nEnrollment rate: {df['enrolled'].mean():.1%}")
print(f"Pell-eligible rate: {df['pell_eligible'].mean():.1%}")
print(f"First-generation rate: {df['first_gen'].mean():.1%}")
print(f"Median AGI: ${df['adjusted_gross_income'].median():,.0f}")
print(f"Median net price: ${df['net_price'].median():,.0f}")
print(f"Median NPIR: {df['net_price_to_income_ratio'].median():.2f}")
print(f"Missing sat_act_percentile (test-optional non-submitters): {df['sat_act_percentile'].isna().mean():.1%}")
print(f"Min unmet_financial_need_gap: {df['unmet_financial_need_gap'].min()} (should be >= 0)")
print(f"Max financial_aid_discount_rate: {df['financial_aid_discount_rate'].max():.4f} (should be <= {MAX_AID_SHARE_OF_COA})")


Enrollment rate: 51.6%
Pell-eligible rate: 34.0%
First-generation rate: 33.9%
Median AGI: $64,302
Median net price: $22,090
Median NPIR: 0.33
Missing sat_act_percentile (test-optional non-submitters): 2.5%
Min unmet_financial_need_gap: 0.0 (should be >= 0)
Max financial_aid_discount_rate: 0.9500 (should be <= 0.95)


In [ ]:
# Display the number of non-null values and its datatype in the dataset
print('\nDataset Info : \n')
print(df.info())

print('\nDataset Info (model-ready) : \n')
print(model_df.info())


Dataset Info : 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 25 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   state_residency              6000 non-null   object 
 1   is_in_state                  6000 non-null   int64  
 2   urban_centric_locale         6000 non-null   object 
 3   student_aid_index            6000 non-null   float64
 4   adjusted_gross_income        6000 non-null   float64
 5   first_gen                    6000 non-null   int64  
 6   pell_eligible                6000 non-null   int64  
 7   hs_gpa                       6000 non-null   float64
 8   sat_act_percentile           5850 non-null   float64
 9   institutional_tier           6000 non-null   object 
 10  cost_of_attendance           6000 non-null   float64
 11  miles_from_campus            6000 non-null   float64
 12  fafsa_submitted_month        6000 non-null   int64  
 13  

In [ ]:
files.download(MODEL_OUTPUT_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>